# NBA Relational Databases, SQL Dumps & Analytical Queries

In [1]:
import os
import glob
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column
from typing import Optional
from datetime import date


## 1. Relational Schema Architecture (SQLAlchemy ORM - 3NF)

In [2]:
class Base(DeclarativeBase):
    pass

class Player(Base):
    __tablename__ = "player"
    player_id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    name: Mapped[str] = mapped_column(nullable=False)
    birth_date: Mapped[Optional[date]]
    height_cm: Mapped[Optional[float]]
    weight_kg: Mapped[Optional[float]]

class College(Base):
    __tablename__ = "college"
    college_id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    name: Mapped[str] = mapped_column(unique=True, nullable=False)

class Team(Base):
    __tablename__ = "team"
    team_id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    abbr: Mapped[str] = mapped_column(unique=True, nullable=False)
    name: Mapped[str] = mapped_column(nullable=False)

class Season(Base):
    __tablename__ = "season"
    season_year: Mapped[int] = mapped_column(primary_key=True)
    label: Mapped[str]
    champion_team_id: Mapped[Optional[int]]

class TeamSeason(Base):
    __tablename__ = "team_season"
    team_id: Mapped[int] = mapped_column(primary_key=True)
    season_year: Mapped[int] = mapped_column(primary_key=True)
    wins: Mapped[Optional[int]]
    losses: Mapped[Optional[int]]
    pace: Mapped[Optional[float]]
    ortg: Mapped[Optional[float]]
    drtg: Mapped[Optional[float]]

class PlayerSeason(Base):
    __tablename__ = "player_season"
    player_season_id: Mapped[int] = mapped_column(primary_key=True, autoincrement=True)
    player_id: Mapped[int]
    team_id: Mapped[Optional[int]]
    season_year: Mapped[int]
    pos: Mapped[Optional[str]]
    exp: Mapped[Optional[int]]
    salary: Mapped[Optional[int]]
    is_total: Mapped[bool] = mapped_column(default=False)


## 2. SQL Dump Files & Scripts Registry

In [3]:
sql_files = [
    "Data Base/load_data.sql",
    "Data Base/nba_db.sql",
    "Data Base/nba_analytics.sql",
    "clean_and_engineered_data/nba_all_clean_and_engineered.sql",
    "cleaned-data/nba_analytics.sql",
    "data/cleaned-data/nba_analytics.sql"
]

sql_registry = []
for path in sql_files:
    if os.path.exists(path):
        size_bytes = os.path.getsize(path)
        size_mb = size_bytes / (1024 * 1024)
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            total_lines = sum(1 for _ in f)
        sql_registry.append({
            "SQL File": os.path.basename(path),
            "Directory": os.path.dirname(path),
            "Size (MB)": round(size_mb, 2),
            "Lines Count": f"{total_lines:,}"
        })

df_sql_registry = pd.DataFrame(sql_registry)
display(df_sql_registry)


,SQL File,Directory,Size (MB),Lines Count
0,load_data.sql,Data Base,0.01,363
1,nba_db.sql,Data Base,5.69,324
2,nba_analytics.sql,Data Base,30.63,"146,655"
3,nba_all_clean_and_engineered.sql,clean_and_engineered_data,44.67,"168,766"
4,nba_analytics.sql,cleaned-data,32.50,"146,662"
5,nba_analytics.sql,data/cleaned-data,32.50,"146,662"


## 3. MySQL Bulk Data Loading Script (`Data Base/load_data.sql`)

In [4]:
load_sql_path = "Data Base/load_data.sql"
with open(load_sql_path, "r", encoding="utf-8") as f:
    load_sql_lines = [next(f) for _ in range(40)]

print(f"File: {load_sql_path}")
print("".join(load_sql_lines))


File: Data Base/load_data.sql
-- ====================================================================
-- MySQL Bulk Data Loader Script (LOAD DATA LOCAL INFILE)
-- Database: nba_db (1980 - 2027)
-- ====================================================================
-- Note: Ensure local_infile is enabled:
--   mysql --local-infile=1 -u root -p
-- ====================================================================

CREATE DATABASE IF NOT EXISTS `nba_db` DEFAULT CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;
USE `nba_db`;

SET @OLD_FOREIGN_KEY_CHECKS=@@FOREIGN_KEY_CHECKS, FOREIGN_KEY_CHECKS=0;
SET @OLD_UNIQUE_CHECKS=@@UNIQUE_CHECKS, UNIQUE_CHECKS=0;
SET @OLD_SQL_MODE=@@SQL_MODE, SQL_MODE='NO_AUTO_VALUE_ON_ZERO';

-- --------------------------------------------------------------------
-- 1. DROP EXISTING TABLES IN REVERSE DEPENDENCY ORDER
-- --------------------------------------------------------------------
DROP TABLE IF EXISTS `player_season_award`;
DROP TABLE IF EXISTS `season_tot

## 4. Standalone MySQL Database Dump (`Data Base/nba_db.sql`)

In [5]:
nba_db_sql_path = "Data Base/nba_db.sql"
with open(nba_db_sql_path, "r", encoding="utf-8", errors="ignore") as f:
    nba_db_lines = [next(f) for _ in range(45)]

print(f"File: {nba_db_sql_path}")
print("".join(nba_db_lines))


File: Data Base/nba_db.sql
-- MySQL dump 10.13  Distrib 8.0.31, for Win64 (x86_64)
-- NBA Complete Relational Database (1980 - 2027)
-- Host: 127.0.0.1    Database: nba_db
-- ------------------------------------------------------
/*!40101 SET @OLD_CHARACTER_SET_CLIENT=@@CHARACTER_SET_CLIENT */;
/*!40101 SET @OLD_CHARACTER_SET_RESULTS=@@CHARACTER_SET_RESULTS */;
/*!40101 SET @OLD_COLLATION_CONNECTION=@@COLLATION_CONNECTION */;
/*!50503 SET NAMES utf8mb4 */;
/*!40103 SET @OLD_TIME_ZONE=@@TIME_ZONE */;
/*!40103 SET TIME_ZONE='+00:00' */;
/*!40014 SET @OLD_UNIQUE_CHECKS=@@UNIQUE_CHECKS, UNIQUE_CHECKS=0 */;
/*!40014 SET @OLD_FOREIGN_KEY_CHECKS=@@FOREIGN_KEY_CHECKS, FOREIGN_KEY_CHECKS=0 */;
/*!40101 SET @OLD_SQL_MODE=@@SQL_MODE, SQL_MODE='NO_AUTO_VALUE_ON_ZERO' */;
/*!40111 SET @OLD_SQL_NOTES=@@SQL_NOTES, SQL_NOTES=0 */;

CREATE DATABASE IF NOT EXISTS `nba_db` DEFAULT CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;
USE `nba_db`;

DROP TABLE IF EXISTS `college`;
CREATE TABLE `college` (
  `

## 5. Normalized Database SQL Dump (`Data Base/nba_analytics.sql`)

In [6]:
nba_analytics_sql = "Data Base/nba_analytics.sql"
with open(nba_analytics_sql, "r", encoding="utf-8", errors="ignore") as f:
    analytics_lines = [next(f) for _ in range(40)]

print(f"File: {nba_analytics_sql}")
print("".join(analytics_lines))


File: Data Base/nba_analytics.sql
BEGIN TRANSACTION;
CREATE TABLE "advanced" (
"Season" INTEGER,
  "Rk" INTEGER,
  "Player" TEXT,
  "Age" REAL,
  "Team" TEXT,
  "Pos" TEXT,
  "G" REAL,
  "GS" REAL,
  "MP" REAL,
  "PER" REAL,
  "TS%" REAL,
  "3PAr" REAL,
  "FTr" REAL,
  "ORB%" REAL,
  "DRB%" REAL,
  "TRB%" REAL,
  "AST%" REAL,
  "STL%" REAL,
  "BLK%" REAL,
  "TOV%" REAL,
  "USG%" REAL,
  "OWS" REAL,
  "DWS" REAL,
  "WS" REAL,
  "WS/48" REAL,
  "OBPM" REAL,
  "DBPM" REAL,
  "BPM" REAL,
  "VORP" REAL,
  "Awards" TEXT,
  "Is_Multi_Team_Aggregate" INTEGER
);
INSERT INTO "advanced" VALUES(1980,1,'Norm Nixon',24.0,'LAL','PG',82.0,82.0,3226.0,14.9,0.548,0.007,0.209,1.9,5.5,3.9,25.3,2.1,0.2,17.9,20.0,4.1,2.7,6.7,0.1,0.2,-0.9,-0.7,1.1,NULL,0);
INSERT INTO "advanced" VALUES(1980,2,'Elvin Hayes',34.0,'WSB','PF',81.0,NULL,3183.0,17.2,0.492,0.008,0.285,8.4,20.0,14.1,6.1,0.9,3.1,10.2,26.1,2.1,4.2,6.2,0.094,-0.2,-0.2,-0.4,1.2,'AS',0);
INSERT INTO "advanced" VALUES(1980,3,'Bill Cartwright',22.0,'NYK','

## 6. Clean & Engineered SQL Dump (`clean_and_engineered_data/nba_all_clean_and_engineered.sql`)

In [7]:
clean_eng_sql = "clean_and_engineered_data/nba_all_clean_and_engineered.sql"
with open(clean_eng_sql, "r", encoding="utf-8", errors="ignore") as f:
    eng_lines = [next(f) for _ in range(40)]

print(f"File: {clean_eng_sql}")
print("".join(eng_lines))


File: clean_and_engineered_data/nba_all_clean_and_engineered.sql
BEGIN TRANSACTION;
CREATE TABLE "advanced_pss" (
"Season" INTEGER,
  "Rk" INTEGER,
  "Player" TEXT,
  "Age" REAL,
  "Team" TEXT,
  "Pos" TEXT,
  "G" REAL,
  "GS" REAL,
  "MP" REAL,
  "PER" REAL,
  "TS%" REAL,
  "3PAr" REAL,
  "FTr" REAL,
  "ORB%" REAL,
  "DRB%" REAL,
  "TRB%" REAL,
  "AST%" REAL,
  "STL%" REAL,
  "BLK%" REAL,
  "TOV%" REAL,
  "USG%" REAL,
  "OWS" REAL,
  "DWS" REAL,
  "WS" REAL,
  "WS/48" REAL,
  "OBPM" REAL,
  "DBPM" REAL,
  "BPM" REAL,
  "VORP" REAL,
  "Awards" TEXT,
  "Is_Multi_Team_Aggregate" INTEGER,
  "WS_Per_Game" REAL,
  "Net_BPM" REAL,
  "Offensive_Load" REAL,
  "Player_Impact_Tier" TEXT,
  "PER_TS_Product" REAL
);
INSERT INTO "advanced_pss" VALUES(1980,1,'Norm Nixon',24.0,'LAL','PG',82.0,82.0,3226.0,14.9,0.548,0.007,0.209,1.9,5.5,3.9,25.3,2.1,0.2,17.9,20.0,4.1,2.7,6.7,0.1,0.2,-0.9,-0.7,1.1,NULL,0,0.0817,1.1,16.42,'Quality Starter',8.17);



## 7. SQLite Relational Database Inspection (`Data Base/nba_analytics.db`)

In [8]:
db_path_1 = "Data Base/nba_analytics.db"
conn1 = sqlite3.connect(db_path_1)
cur1 = conn1.cursor()

tables1 = [r[0] for r in cur1.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()]
print(f"Database: {db_path_1}")
print("-" * 55)
for t in tables1:
    cnt = cur1.execute(f"SELECT COUNT(*) FROM '{t}';").fetchone()[0]
    print(f"Table: {t:<30} Count: {cnt:,}")


Database: Data Base/nba_analytics.db
-------------------------------------------------------
Table: player_master_features         Count: 20,848
Table: pg_mjt_candidates              Count: 69
Table: players                        Count: 5,416
Table: rosters                        Count: 21,435
Table: totals                         Count: 25,671
Table: per_game                       Count: 25,671
Table: advanced                       Count: 25,671
Table: mjt                            Count: 739
Table: salaries                       Count: 18,758
Table: seasons                        Count: 90
Table: teams                          Count: 129
Table: teams_all_season               Count: 1,848


## 8. Analytical SQLite Database (`clean_and_engineered_data/nba_all_clean_and_engineered.db`)

In [9]:
db_path_2 = "clean_and_engineered_data/nba_all_clean_and_engineered.db"
conn2 = sqlite3.connect(db_path_2)
cur2 = conn2.cursor()

tables2 = [r[0] for r in cur2.execute("SELECT name FROM sqlite_master WHERE type='table';").fetchall()]
print(f"Database: {db_path_2}")
print("-" * 55)
for t in tables2:
    cnt = cur2.execute(f"SELECT COUNT(*) FROM '{t}';").fetchone()[0]
    print(f"Table: {t:<35} Count: {cnt:,}")


Database: clean_and_engineered_data/nba_all_clean_and_engineered.db
-------------------------------------------------------
Table: players                             Count: 5,416
Table: rosters                             Count: 21,435
Table: totals_pss                          Count: 25,671
Table: per_game_pss                        Count: 25,671
Table: advanced_pss                        Count: 25,671
Table: micheal_jordan_trophy               Count: 739
Table: rosters_salaries                    Count: 18,758
Table: seasons                             Count: 90
Table: teams                               Count: 129
Table: teams_all_season                    Count: 1,848
Table: cross_player_master_features        Count: 26,132
Table: cross_pg_mjt_scouting_candidates    Count: 851
Table: cross_salary_vs_performance         Count: 15,822
Table: cross_champions_vs_top_players      Count: 47
Table: cross_agility_innate_hypothesis     Count: 48


## 9. Analytical SQL Queries with Joins (Phase 2 Requirement)

In [10]:
query_totals_adv = """
SELECT 
    t.Season,
    t.Player,
    t.Pos,
    t.G,
    t.MP,
    t.PTS,
    t.AST,
    t.TRB,
    a.PER,
    a.WS,
    a.BPM,
    a.VORP
FROM totals_pss t
INNER JOIN advanced_pss a 
    ON t.Season = a.Season 
    AND t.Player = a.Player 
    AND t.Team = a.Team
WHERE t.Season >= 2020 AND t.G >= 20
ORDER BY t.PTS DESC
LIMIT 10;
"""

df_top_performers = pd.read_sql_query(query_totals_adv, conn2)
display(df_top_performers)


,Season,Player,Pos,G,MP,PTS,AST,TRB,PER,WS,BPM,VORP
0,2025,Shai Gilgeous-Alexander,PG,76.0,2598.0,2484.0,486.0,379.0,30.7,16.7,11.5,8.9
1,2024,Luka Dončić,PG,70.0,2624.0,2370.0,686.0,647.0,28.1,12.0,9.9,8.0
2,2020,James Harden,SG,68.0,2483.0,2335.0,512.0,446.0,29.1,13.1,9.6,7.3
3,2024,Shai Gilgeous-Alexander,PG,75.0,2553.0,2254.0,465.0,415.0,29.3,14.6,9.0,7.1
4,2023,Jayson Tatum,SF,74.0,2732.0,2225.0,342.0,649.0,23.7,10.5,5.5,5.1
5,2024,Giannis Antetokounmpo,PF,73.0,2567.0,2222.0,476.0,841.0,29.9,13.2,9.0,7.2
6,2024,Jalen Brunson,PG,77.0,2726.0,2212.0,519.0,278.0,23.4,11.2,5.8,5.4
7,2023,Joel Embiid,C,66.0,2284.0,2183.0,274.0,670.0,31.4,12.3,9.2,6.4
8,2025,Anthony Edwards,SG,79.0,2871.0,2177.0,359.0,450.0,20.1,8.4,4.3,4.6
9,2022,Trae Young,PG,76.0,2652.0,2155.0,737.0,284.0,25.4,10.0,5.2,4.8


## 10. SQL Query: Height Comparison (MJT Candidates vs Top Scorers)

In [11]:
query_mjt_height = """
SELECT 
    m.Season,
    m.Player,
    m.Rank,
    m.Share,
    p.Ht_CM,
    p.Wt_KG,
    p.Agility_Metric
FROM micheal_jordan_trophy m
LEFT JOIN players p ON m.Player = p.Player
WHERE m.Season BETWEEN 2020 AND 2024
ORDER BY m.Season DESC, m.Share DESC
LIMIT 10;
"""

df_mjt_sql = pd.read_sql_query(query_mjt_height, conn2)
display(df_mjt_sql)


,Season,Player,Rank,Share,Ht_CM,Wt_KG,Agility_Metric
0,2024,Nikola Jokić,1,0.935,210.82,128.82,1.6365
1,2024,Shai Gilgeous-Alexander,2,0.646,198.12,88.45,2.2399
2,2024,Luka Dončić,3,0.572,203.20,104.33,1.9477
3,2024,Giannis Antetokounmpo,4,0.194,210.82,113.40,1.8591
4,2024,Jalen Brunson,5,0.143,187.96,86.18,2.1810
5,2024,Jayson Tatum,6,0.087,203.20,95.25,2.1333
6,2024,Anthony Edwards,7,0.018,193.04,102.06,1.8914
7,2024,Domantas Sabonis,8,0.003,208.28,108.86,1.9133
8,2024,Kevin Durant,9,0.001,210.82,108.86,1.9366
9,2023,Joel Embiid,1,0.915,213.36,122.47,1.7421


## 11. SQL Query: Point Guard Scouting Ranking

In [12]:
query_pg_scouting = """
SELECT 
    Player,
    Seasons_Played,
    MJT_Appearances,
    Total_MVP_Share,
    Avg_PTS_Per_36,
    Avg_AST_Per_36,
    Avg_AST_TOV_Ratio,
    Scouting_Score,
    Scouting_Rank
FROM cross_pg_mjt_scouting_candidates
WHERE Scouting_Rank <= 10
ORDER BY Scouting_Rank ASC;
"""

df_pg_sql = pd.read_sql_query(query_pg_scouting, conn2)
display(df_pg_sql)


,Player,Seasons_Played,MJT_Appearances,Total_MVP_Share,Avg_PTS_Per_36,Avg_AST_Per_36,Avg_AST_TOV_Ratio,Scouting_Score,Scouting_Rank
0,LeBron James,2,20,8.834,26.7,9.5,2.35,877.30,1
1,Magic Johnson,8,12,5.104,19.8,12.1,3.16,529.22,2
2,James Harden,8,10,3.656,23.8,9.5,2.44,416.09,3
3,Giannis Antetokounmpo,1,9,4.175,17.2,4.4,1.66,409.12,4
4,Stephen Curry,18,11,2.666,27.2,6.5,1.94,384.76,5
5,Chris Paul,21,11,1.807,17.5,9.8,4.05,339.03,6
6,Steve Nash,18,8,2.428,15.5,9.4,2.88,307.90,7
7,John Stockton,19,12,0.163,14.6,11.7,3.68,277.41,8
8,Russell Westbrook,17,8,1.619,22.5,8.7,2.05,271.92,9
9,Allen Iverson,7,8,1.567,21.8,5.9,1.81,264.56,10


In [13]:
conn1.close()
conn2.close()
print("All database connections closed successfully.")


All database connections closed successfully.
